In [ ]:
from torchvision import datasets, transforms




dataset = datasets.ImageNet(
            root=f"~/FuXianHu/datasets/ImageNet64_HF",
            split='train',
            transform= transforms.Compose([
                transforms.Resize(64),
                transforms.CenterCrop(64),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ])
        )


In [4]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class ImageNet64(Dataset):
  def __init__(self, split="train", transform=None):
      self.dataset = load_dataset(
          "benjamin-paine/imagenet-1k-64x64",
          cache_dir="/home/magicyu/fuxianhu/datasets/ImageNet64_HF",
          split=split
      )
      self.transform = transform

  def __len__(self):
      return len(self.dataset)

  def __getitem__(self, idx):
      sample = self.dataset[idx]
      image = sample["image"].convert("RGB")
      label = sample["label"]
      if self.transform:
          image = self.transform(image)
      return image, label


# Usage
transform = transforms.Compose([
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225]),
])

train_dataset = ImageNet64(split="train", transform=transform)
val_dataset   = ImageNet64(split="validation", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=128, shuffle=False, num_workers=4)


In [8]:
dataset = load_dataset("benjamin-paine/imagenet-1k-64x64",
                        cache_dir="/home/magicyu/fuxianhu/datasets/ImageNet64_HF",
                        split="train"
                      )

In [6]:
train_dataset.__getitem__(0)

(tensor([[[2.2318, 2.1462, 2.1633,  ..., 2.2489, 2.2489, 2.1804],
          [2.2318, 2.1975, 2.2318,  ..., 2.1975, 2.2489, 2.2489],
          [2.2318, 2.2489, 2.2489,  ..., 2.1975, 2.2489, 2.2489],
          ...,
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          [2.2489, 2.2489, 2.2489,  ..., 2.2318, 2.2489, 2.2489]],
 
         [[2.4286, 2.3585, 2.4111,  ..., 2.4286, 2.4286, 2.3585],
          [2.4286, 2.4111, 2.4286,  ..., 2.3761, 2.4286, 2.4286],
          [2.4111, 2.4286, 2.4286,  ..., 2.3761, 2.4286, 2.4286],
          ...,
          [2.4111, 2.3936, 2.4111,  ..., 2.4111, 2.4111, 2.4111],
          [2.4111, 2.3761, 2.4111,  ..., 2.4286, 2.4111, 2.4286],
          [2.4111, 2.3761, 2.4111,  ..., 2.3936, 2.3761, 2.4111]],
 
         [[2.6051, 2.5180, 2.6400,  ..., 2.6400, 2.6051, 2.5354],
          [2.6051, 2.5703, 2.6400,  ..., 2.5877, 2.6051, 2.6051],
          [2.5877, 2.6051, 2.6400,  ...,

In [23]:
dataset[0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=64x64>,
 'label': 0}

In [62]:
from typing import Optional, List, Type, Tuple, Dict
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
from trainer import Trainer
from flow_match import GaussianConditionalProbabilityPath
from simulate import ConditionalVectorField, CFGVectorFieldODE, EulerSimulator

from data_sampler import Sampleable
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from flow_match import LinearBeta, LinearAlpha
from model import MNISTUNet

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


class ImageNetSampler(nn.Module, Sampleable):
    def __init__(self):
        super().__init__()
        self.dataset = load_dataset("benjamin-paine/imagenet-1k-64x64",
                                    cache_dir="/home/magicyu/fuxianhu/datasets/ImageNet64_HF",
                                    split="train"
                                  )

        self.transform = transforms.Compose([
                                            transforms.Resize(64),
                                            transforms.CenterCrop(64),
                                            transforms.ToTensor(),
                                            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                                        ])

        self.dummy = nn.Buffer(torch.zeros(1))  # Will automatically be moved when self.to(...) is called...

    def sample(self, num_samples: int) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Args:
            - num_samples: the desired number of samples
        Returns:
            - samples: shape (batch_size, c, h, w)
            - labels: shape (batch_size, label_dim)
        """
        if num_samples > len(self.dataset):
            raise ValueError(f"num_samples exceeds dataset size: {len(self.dataset)}")

class ImageNetSampler(nn.Module, Sampleable):
    def __init__(self):
        super().__init__()
        self.dataset = load_dataset("benjamin-paine/imagenet-1k-64x64",
                                    cache_dir="/home/magicyu/fuxianhu/datasets/ImageNet64_HF",
                                    split="train"
                                  )

        self.transform = transforms.Compose([
                                            transforms.Resize(64),
                                            transforms.CenterCrop(64),
                                            transforms.ToTensor(),
                                            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                                        ])

        # self.dummy = nn.Buffer(torch.zeros(1))  # Will automatically be moved when self.to(...) is called...
        self.register_buffer("dummy", torch.zeros(1))

    def sample(self, num_samples: int) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Args:
            - num_samples: the desired number of samples
        Returns:
            - samples: shape (batch_size, c, h, w)
            - labels: shape (batch_size, label_dim)
        """
        if num_samples > len(self.dataset):
            raise ValueError(f"num_samples exceeds dataset size: {len(self.dataset)}")

        indices = torch.randperm(len(self.dataset))[:num_samples].tolist()
        batch = self.dataset[indices]  # single batched fetch

        samples = torch.stack([self.transform(img.convert("RGB")) for img in batch["image"]])
        labels = torch.tensor(batch["label"], dtype=torch.int64)

        return samples.to(self.dummy), labels.to(self.dummy.device)

In [63]:
imagenet_sampler = ImageNetSampler()



In [66]:
image[0].shape

torch.Size([3, 64, 64])